# Atelier ML - Apprentissage : Partie 3 (prédictions)

## Utiliser le modèle d'apprentissage dans une application
On dispose d'un modèle capable de prédire le pourcentage de victoire de chaque Pokémon (sauvegardé par `partie3_entrainement.ipynb` dans `modele/modele_pokemon.mod`). On l'utilise ici pour **prédire le vainqueur d'un combat** entre deux Pokémons du Pokédex

(Executez `partie3_entrainement.ipynb` avant de commencer cet atelier)

In [15]:
#Module de lecture de fichiers CSV
import csv
#Module de chargement du modèle d'apprentissage
import joblib

Nous allons ensuite écrire une première fonction qui sera chargée de chercher les informations d’un Pokémon dans le Pokédex à partir de son numéro et utiles à notre modèle de prédiction

In [16]:
def rechercheInformationsPokemon(numPokemon,Pokedex):
    infosPokemon = []
    for pokemon in Pokedex:
        if (int(pokemon[0])==numPokemon):
            infosPokemon = [pokemon[0],pokemon[1],pokemon[4],pokemon[5],pokemon[6],pokemon[7],pokemon[8],pokemon[9],pokemon[10]]
            break
    return infosPokemon

1. Creation d'une liste `infosPokemon` qui stocke toutes les informations du Pokemon necessaires a la prediction de sa victoire/defaite
2. Parcours du Pokédex à la recherche du Pokémon ayant le même numéro que celui que nous cherchons (le numéro du Pokémon correspondant à la première colonne du Pokédex est transformé en entier `(int(pokemon[0]))` afin de pouvoir le comparer avec le numéro du Pokémon recherché, passé en paramètre de la fonction)
3. Si lors de la boucle de parcours du Pokédex, le numéro lu correspond au numéro du Pokémon recherché, on extrait alors les informations nécessaires pour pouvoir ensuite les renvoyer à la fonction appelante de notre fonction de recherche

Le tableau ci-dessous reprend l’information nécessaire à la prédiction de victoire ou de défaite et sa position dans le Pokédex (le numéro se trouve dans la première colonne (pokemon[0]), le nom dans la deuxième (pokemon[1])…)


| Information | Indice de colonne dans le Pokédex |
|---|---|
| Numéro | 0 |
| Nom | 1 |
| Points de vie | 4 |
| Niveau d'attaque | 5 |
| Niveau de défense | 6 |
| Niveau d'attaque spécial | 7 |
| Niveau de défense spécial | 8 |
| Vitesse | 9 |
| Génération | 10 |

## Fonction de prédiction
Elle prend en paramètres le numéro du premier Pokémon, celui du second, et le Pokédex. Elle :
1. recherche les informations des deux Pokémons,
2. charge le modèle,
3. prédit le pourcentage de victoire de chacun,
4. déclare vainqueur celui qui a le pourcentage le plus élevé.

In [17]:
def prediction (numeroPokemon1, numeroPokemon2,Pokedex):
    pokemon1 = rechercheInformationsPokemon(numeroPokemon1, Pokedex)
    pokemon2 = rechercheInformationsPokemon(numeroPokemon2, Pokedex)
    
    modele_prediction = joblib.load('modele/modele_pokemon.mod')
    
    prediction_Pokemon_1 = modele_prediction.predict([[pokemon1[2],pokemon1[3],pokemon1[4],pokemon1[5],pokemon1[6],pokemon1[7],pokemon1[8]]])
    
    prediction_Pokemon_2 = modele_prediction.predict([[pokemon2[2], pokemon2[3], pokemon2[4], pokemon2[5], pokemon2[6], pokemon2[7], pokemon2[8]]])
    
    print ("COMBAT OPPOSANT : ("+str(numeroPokemon1)+") "+pokemon1[1]+" à ("+str(numeroPokemon2)+") "+pokemon2[1])
    print (" "+pokemon1[1]+": "+str(prediction_Pokemon_1[0]))
    print(" " +pokemon2[1] + ": " + str(prediction_Pokemon_2[0]))
    print ("")
    
    if prediction_Pokemon_1>prediction_Pokemon_2:
        print(pokemon1[1].upper()+" EST LE VAINQUEUR !")
    else:
        print(pokemon2[1].upper() + " EST LE VAINQUEUR !")


#chargement du modele et la prediction pour chaque Pokemon
modele_prediction = joblib.load('modele/modele_pokemon.mod')

On compare ensuite le pourcentage de victoire prédit pour en déduire le vainqueur du combat.

Vient enfin le moment de saisir les dernières lignes de code, en chargeant le fichier Pokédex, en excluant sa première ligne contenant les noms des différentes colonnes `(next(pokedex))` et en appelant la fonction de prédiction

In [18]:
#Chargement du Pokédex et lancement d'un combat
with open("data/pokedex.csv", newline='') as csvfile:
    pokedex = csv.reader(csvfile)
    next(pokedex)
    prediction(368,598,pokedex)

COMBAT OPPOSANT : (368) Mangriff à (598) Crapustule
 Mangriff: 96.13
 Crapustule: 75.02

MANGRIFF EST LE VAINQUEUR !


## Testing avec combats.csv

In [19]:
import pandas as pd
combats = pd.read_csv("data/combats.csv")
modele = joblib.load('modele/modele_pokemon.mod')

with open("data/pokedex.csv", newline='', encoding='utf-8') as csvfile:
    lecteur = csv.reader(csvfile)
    next(lecteur)
    pokedexListe = list(lecteur)
# Informations utiles à la prédiction, indexées par numéro de Pokémon (7 features)
infos = {int(p[0]): [float(v) for v in p[4:11]] for p in pokedexListe}

nbCombatsTestes = 1000
echantillon = combats.head(nbCombatsTestes)

bonnesPredictions = 0
for _, ligne in echantillon.iterrows():
    p1, p2, gagnant = int(ligne['Premier_Pokemon']), int(ligne['Second_Pokemon']), int(ligne['Pokemon_Gagnant'])
    score1 = modele.predict([infos[p1]])[0]
    score2 = modele.predict([infos[p2]])[0]
    vainqueurPredit = p1 if score1 > score2 else p2
    if vainqueurPredit == gagnant:
        bonnesPredictions += 1

print(f"Bonnes prédictions : {bonnesPredictions} / {nbCombatsTestes} ({bonnesPredictions / nbCombatsTestes:.1%})")

Bonnes prédictions : 870 / 1000 (87.0%)
